In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim ,length
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType

In [0]:
df=spark.table("bronze.erp_cust_az12")
df.display()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df=df.withColumn(field.name,trim(col(field.name)))

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)
     

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.show(5)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_cust_az12")

In [0]:
%sql
select * from silver.erp_cust_az12 limit 5